In [1]:
import os
import glob
import pandas as pd
import json
from pathlib import Path
from typing import Dict, List, Any


FLOW_SOURCE_DIR = "../../backend/source/akvo-flow"
FORMS_SOURCE_DIR = "../../backend/source/forms"
# Column names for question mapping
FLOW_QUESTION_ID_COL = "flow_question_id"
MIS_QUESTION_ID_COL = "mis_question_id"
MIS_FORM_ID_COL = "mis_form_id"


def load_data_file(
    flow_id: int
) -> pd.DataFrame:
    """
    Load data CSV file for the specified Flow form ID.

    Args:
        flow_id: The Akvo Flow form ID

    Returns:
        DataFrame with data or None if file not found/invalid
    """
    csv_path = os.path.join(FLOW_SOURCE_DIR, "data", f"{flow_id}_*.csv")
    # Find matching file (handles versioning like _v1, _v2)
    try:
        matching_files = glob.glob(csv_path)
        if not matching_files:
            return None
        csv_path = matching_files[0]  # Use first match
        df = pd.read_csv(
            csv_path,
            encoding="utf-8",
            low_memory=False,  # Better for large files
        )
        return df
    except Exception as e:
        print(f"Error loading data file for Flow ID {flow_id}: {e}")
        return None

def load_question_mappings(
    flow_id: int
) -> Dict[str, List[str]]:
    """
    Load question mappings from CSV file.

    Args:
        flow_id: The Akvo Flow form ID

    Returns:
        Dictionary mapping flow_question_id to list of mis_question_ids
    """
    csv_path = os.path.join(
        FLOW_SOURCE_DIR, f"{flow_id}_forms_mapping.csv"
    )
    try:
        df = pd.read_csv(
            csv_path,
            encoding="utf-8",
            dtype={FLOW_QUESTION_ID_COL: str, MIS_QUESTION_ID_COL: str},
        )
        # group by mis_form_id
        df_grouped = df.groupby(MIS_FORM_ID_COL)
        return [
            {
                MIS_FORM_ID_COL: mis_form_id,
                "questions": group_df[[FLOW_QUESTION_ID_COL, MIS_QUESTION_ID_COL]]
                .dropna()
                .query(
                    f"{MIS_QUESTION_ID_COL} != '' and {FLOW_QUESTION_ID_COL} != ''"
                )
                .apply(
                    lambda row: {
                        FLOW_QUESTION_ID_COL: row[FLOW_QUESTION_ID_COL],
                        MIS_QUESTION_ID_COL: row[MIS_QUESTION_ID_COL],
                    },
                    axis=1,
                )
                .tolist(),
            }
            for mis_form_id, group_df in df_grouped
        ]
    except Exception as e:
        print(f"Error loading question mappings for Flow ID {flow_id}: {e}")
        return {}

def load_json_form(
    mis_form_id: str
):
    # Find MIS form JSON
    json_path = next(filter(
        lambda f: f.suffix == ".json" and mis_form_id in f.name,
        Path(FORMS_SOURCE_DIR).iterdir(),
    ), None)
    if not json_path:
        return []
    j = pd.read_json(json_path, typ="series").to_dict()
    df = pd.json_normalize(
        j,
        record_path=["question_groups", "questions"],
        meta= [
            "id",
            "form",
            "type",
            "version",
            ["question_groups", "label"],
            ["question_groups", "repeatable"],
        ],
        meta_prefix=f"mis_",
        errors="ignore"
    )
    df = df.rename(
        columns={
            "mis_question_groups.label": "question_group",
            "mis_question_groups.repeatable": "question_group_repeatable"
        }
    )
    return df.to_dict(orient="records")

def normalize_value(value: Any) -> Any:
    """
    Normalize various value types to string format for database storage.
    Args:
        value: The input value to normalize.
    """
    if pd.isna(value):
        return ""

    # If it's a JSON string (common in Flow exports)
    if isinstance(value, str) and (
        value.startswith("{") or value.startswith("[")
    ):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, str):
                # If the parsed value is still a string, normalize it
                return normalize_value(parsed)
            return parsed
        except json.JSONDecodeError:
            return str(value).strip()
    return value

def transform_mis_value(
    questions: List[dict],
    question_id: str,
    answer: str = None
):
    answer = normalize_value(answer)
    # find questions by question_id
    fq = list(filter(lambda q: str(q["id"]) == question_id, questions))
    if len(fq) == 0:
        return answer
    question = fq[0]
    if question["type"] == "administration" and isinstance(answer, list):
        return " - ".join([
            a["name"] if "name" in a else a["text"]
            for a in answer
        ])
    if question["type"] == "geo" and isinstance(answer, dict):
        lat = answer.get("lat", "")
        lon = answer.get("long", "")
        return f"{lat}|{lon}"
    if question["type"] in [
        "photo",
        "signature",
    ]:
        if "image" in answer:
            return f"data:image/png;base64,{answer['image']}"
        if "filename" in answer:
            return answer["filename"]
        if isinstance(answer, list):
            return "|".join([
              f"data:image/png;base64,{a['image']}"
                for a in answer
                if "image" in a
            ])
    if isinstance(answer, list):
        val_list = []
        for a in answer:
            if "text" in a:
                val_list.append(a["text"])
            if "isOther" in a and a["isOther"] == True:
                val_list.append(a.get("otherText", "other"))
        return "|".join(val_list)
    if isinstance(answer, dict) and "text" in answer:
        print("dict answer", answer)
        # return answer["text"]
    return answer

In [2]:
flow_form_id = '1520924'
flow_ids = {
    "8520967": 1749634736797, # WTP
    "17260923": 1748903240763, # WWTP
    "27040920": 1749611049520, # SPS (Pump Station)
    "1520924": 1749623934933, # EPS Water quality
    "5530933": 1749623934933, # EPS Project Construction
    "2490944": 1749621221728, # RWS
}
data_df = load_data_file(flow_form_id)
seed_questions = load_question_mappings(flow_form_id)
mis_form_id = flow_ids[flow_form_id]
mis_questions = load_json_form(mis_form_id=str(mis_form_id))
meta_questions = list(filter(
    lambda  q: ((q["meta"] is True or q["meta"] == "true") and q["type"] != "geo"),
    mis_questions
))
meta_question_ids = [str(mq["id"]) for mq in meta_questions]

In [3]:
parent_data = []
child_data = []
"""
parent data is from mis parent form
child data is from mis child forms
"""
for idx, row in data_df.iterrows():
    for fq in seed_questions:
        f_qs = load_json_form(mis_form_id=str(int(fq[MIS_FORM_ID_COL])))
        questions = list(filter(lambda q: q[FLOW_QUESTION_ID_COL] in row, fq["questions"]))
        row_data = {
            q[MIS_QUESTION_ID_COL]: transform_mis_value(
                questions=f_qs,
                question_id=q[MIS_QUESTION_ID_COL],
                answer=row[q[FLOW_QUESTION_ID_COL]]
            )
            for q in questions
            if pd.notna(row[q[FLOW_QUESTION_ID_COL]])
        }
        if fq[MIS_FORM_ID_COL] == mis_form_id:
            meta_available = list(filter(
                lambda m: m,
                [
                    row_data[mq] if mq in row_data else None
                    for mq in meta_question_ids
                ]
            ))
            # meta_available = [row["displayName"]] + meta_available
            meta_name = " - ".join(meta_available)
            parent_data.append({
                "form_id": mis_form_id,
                "identifier": row["identifier"],
                "created_at": row["createdAt"],
                "datapoint_id": row["datapoint_id"],
                "name": meta_name,
                **row_data
            })
        else:
            # Filter out empty rows
            if any(v not in [None, ""] for v in row_data.values()):
                child_data.append({
                    "form_id": fq[MIS_FORM_ID_COL],
                    "identifier": row["identifier"],
                    "created_at": row["createdAt"],
                    "datapoint_id": row["datapoint_id"],
                    "name": row["displayName"],
                    **row_data
                })

In [4]:
parent_data_df = pd.DataFrame(parent_data)
child_data_df = pd.DataFrame(child_data)

In [5]:
# Reduce duplication in parent_data_df by checking unique name
parent_data_df = parent_data_df.drop_duplicates(subset=["name"])
# convert parent_df and child_df to CSV files
parent_data_df.to_csv(
    os.path.join(FLOW_SOURCE_DIR, "output", f"{flow_form_id}_parent_data.csv"),
    index=False
)
print("total parent records:", len(parent_data_df))
print("parent data saved")

# Filter child_data_df based on datapoint_id in parent_data_df
child_data_df = child_data_df[
    child_data_df["datapoint_id"].isin(parent_data_df["datapoint_id"])
]
print("total child records after filtering:", len(child_data_df))
child_data_df.to_csv(
    os.path.join(FLOW_SOURCE_DIR, "output", f"{flow_form_id}_child_data.csv"),
    index=False
)
print("child data saved")

total parent records: 252
parent data saved
total child records after filtering: 504
child data saved
